In [11]:
# =========================================================
# 0. IMPORTS
# =========================================================

import os
import io
import random
import warnings
import contextlib

warnings.filterwarnings("ignore")

# =========================================================
# GLOBAL SEED / DETERMINISM
# =========================================================

SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["TF_DETERMINISTIC_OPS"] = "1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

# =========================================================
# CORE
# =========================================================

import joblib
import numpy as np
import pandas as pd

random.seed(SEED)
np.random.seed(SEED)

# =========================================================
# METRICS
# =========================================================

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# =========================================================
# MODELS
# =========================================================

import lightgbm as lgb
from xgboost import XGBRegressor

# =========================================================
# PREPROCESSING
# =========================================================

from sklearn.preprocessing import StandardScaler

# =========================================================
# NEURAL NETS
# =========================================================

import tensorflow as tf

tf.get_logger().setLevel("ERROR")
tf.keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.regularizers import l2

In [ ]:
# =========================================================
# 1. GLOBAL SETTINGS
# =========================================================

# time resolution and target variable
time_resolution = "qh"   # "qh" or "h"
target_col = "id1"       # "id1" or "id3"

# models used in the comparison
selected_models = ["lgb", "xgb", "nn"]

# rolling setup
test_year = 2025
rolling_train_months = 12

# lag settings
fundamental_lags = [1, 2]

In [13]:
# =========================================================
# 2. PREDICTOR DEFINITIONS
# =========================================================

# current, non-lagged predictors used by the model
predictor_vars = [
    "day_ahead_price",
    "load_actual_mw",
    "load_delta",
    "wind_delta",
    "solar_delta",
    "import_delta",
    "export_delta",
    "net_import_total",
    "fossil_gas_mw",
    "biomass_mw",
    "hydro_run_of_river_and_poundage_mw",
    "hydro_water_reservoir_mw",
    "hydro_pumped_storage_mw",
    "solar_mw",
    "wind_onshore_mw",
    "outage_total_true",
    "ramp_outage",
]

# subset of predictors for which lagged versions are added
lagged_fundamental_vars = [
    "load_delta",
    "wind_delta",
    "solar_delta",
    "net_import_total",
    "ramp_outage",
]

time_features = [
    "hour_sin",
    "hour_cos",
    "month_sin",
    "month_cos",
    "free_day",
]

In [24]:
# =========================================================
# 3. HELPER FUNCTIONS
# =========================================================

@contextlib.contextmanager
def suppress_training_output():
    with contextlib.redirect_stdout(io.StringIO()):
        with contextlib.redirect_stderr(io.StringIO()):
            yield



def add_lag_features(df, columns, lags):
    df = df.copy()

    for col in columns:
        if col not in df.columns:
            continue

        for lag in lags:
            df[f"{col}_lag{lag}"] = df[col].shift(lag)

    return df


def build_feature_columns(df, predictor_vars, lagged_fundamental_vars, fundamental_lags, time_features):
    current_features = [col for col in predictor_vars if col in df.columns]

    lagged_features = [
        f"{col}_lag{lag}"
        for col in lagged_fundamental_vars
        for lag in fundamental_lags
        if f"{col}_lag{lag}" in df.columns
    ]

    available_time_features = [col for col in time_features if col in df.columns]

    feature_cols = current_features + lagged_features + available_time_features
    return list(dict.fromkeys(feature_cols))


def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))


def evaluate_price_metrics(actual_price, pred_price):
    residuals = np.asarray(actual_price) - np.asarray(pred_price)
    abs_residuals = np.abs(residuals)

    return {
        "ResidualMean": np.mean(residuals),
        "MAE": mean_absolute_error(actual_price, pred_price),
        "RMSE": rmse(actual_price, pred_price),
        "R2": r2_score(actual_price, pred_price),
        "p95_abs_residual": np.percentile(abs_residuals, 95),
        "p99_abs_residual": np.percentile(abs_residuals, 99),
    }


def make_random_validation_split(X_train, y_train, valid_fraction=0.15, seed=SEED):
    n_valid = int(np.floor(len(X_train) * valid_fraction))

    if n_valid < 1:
        raise ValueError("Validation set would be empty. Increase training data or valid_fraction.")

    rng = np.random.RandomState(seed)
    val_idx = rng.choice(X_train.index.to_numpy(), size=n_valid, replace=False)

    X_val = X_train.loc[val_idx].copy()
    y_val = y_train.loc[val_idx].copy()
    X_tr = X_train.drop(val_idx).copy()
    y_tr = y_train.drop(val_idx).copy()

    return X_tr, y_tr, X_val, y_val


def compute_mean_bias(y_true, y_pred):
    return float(np.mean(np.asarray(y_true) - np.asarray(y_pred)))


def apply_bias_correction(y_pred, bias_correction):
    return np.asarray(y_pred) + bias_correction


def format_metric_table(df_metrics):
    ordered_cols = [
        "Model",
        "ResidualMean",
        "MAE",
        "RMSE",
        "R2",
        "MeanError",
        "p95_abs_residual",
        "p99_abs_residual",
    ]

    existing_cols = [
        col for col in ordered_cols
        if col in df_metrics.columns
    ]

    return df_metrics[existing_cols].reset_index(drop=True)


def build_id1_hat_frame(datetimes, pred_price):
    return (
        pd.DataFrame({
            "datetime": pd.to_datetime(datetimes),
            f"{target_col.upper()}_hat": np.asarray(pred_price),
        })
        .sort_values("datetime")
        .reset_index(drop=True)
    )


In [15]:
# =========================================================
# 4. LOAD DATA
# =========================================================

df = pd.read_csv(
    f"../data/combined_data_{time_resolution}.csv",
    parse_dates=["datetime"],
)

df["datetime"] = pd.to_datetime(df["datetime"], utc=True).dt.tz_convert("Europe/Vienna")
df = df.sort_values("datetime").reset_index(drop=True)
df = df.loc[:, ~df.columns.duplicated()].copy()

print("data loaded:", df.shape)
print("start:", df["datetime"].min())
print("end:  ", df["datetime"].max())

data loaded: (140256, 132)
start: 2022-01-01 00:00:00+01:00
end:   2025-12-31 23:45:00+01:00


In [16]:
# =========================================================
# 5. CREATE TARGET AND LAGGED FEATURES
# =========================================================

spread_col = f"spread_{target_col}"
df[spread_col] = df[target_col] - df["day_ahead_price"]

df = add_lag_features(df, lagged_fundamental_vars, fundamental_lags)

print("target:", spread_col)
print("lags:  ", fundamental_lags)

target: spread_id1
lags:   [1, 2]


In [17]:
# =========================================================
# 6. BUILD FEATURE LIST
# =========================================================

feature_cols = build_feature_columns(
    df=df,
    predictor_vars=predictor_vars,
    lagged_fundamental_vars=lagged_fundamental_vars,
    fundamental_lags=fundamental_lags,
    time_features=time_features,
)

print(f"n_features = {len(feature_cols)}")
print(feature_cols)

n_features = 32
['day_ahead_price', 'load_actual_mw', 'load_delta', 'wind_delta', 'solar_delta', 'import_delta', 'export_delta', 'net_import_total', 'fossil_gas_mw', 'biomass_mw', 'hydro_run_of_river_and_poundage_mw', 'hydro_water_reservoir_mw', 'hydro_pumped_storage_mw', 'solar_mw', 'wind_onshore_mw', 'outage_total_true', 'ramp_outage', 'load_delta_lag1', 'load_delta_lag2', 'wind_delta_lag1', 'wind_delta_lag2', 'solar_delta_lag1', 'solar_delta_lag2', 'net_import_total_lag1', 'net_import_total_lag2', 'ramp_outage_lag1', 'ramp_outage_lag2', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'free_day']


In [18]:
# =========================================================
# 7. BUILD MODELING DATAFRAME AND ROLLING WINDOWS
# =========================================================

needed_cols = ["datetime", target_col, "day_ahead_price", spread_col] + feature_cols
needed_cols = list(dict.fromkeys([col for col in needed_cols if col in df.columns]))

df_model = df[needed_cols].dropna().copy()
df_model["year_month"] = df_model["datetime"].dt.to_period("M")

test_months = pd.period_range(f"{test_year}-01", f"{test_year}-12", freq="M")
rolling_windows = []

for test_month in test_months:
    train_start = test_month - rolling_train_months
    train_end = test_month - 1

    train_mask = (df_model["year_month"] >= train_start) & (df_model["year_month"] <= train_end)
    test_mask = df_model["year_month"] == test_month

    train_window_df = df_model.loc[train_mask].copy()
    test_window_df = df_model.loc[test_mask].copy()

    if train_window_df.empty or test_window_df.empty:
        print(f"skipping {test_month}: missing train or test data")
        continue

    rolling_windows.append({
        "test_month": str(test_month),
        "train_start": str(train_start),
        "train_end": str(train_end),
        "train_df": train_window_df,
        "test_df": test_window_df,
    })

print("df_model shape:", df_model.shape)
print("df_model period:", df_model["datetime"].min(), "->", df_model["datetime"].max())
print("n rolling windows:", len(rolling_windows))

for window in rolling_windows:
    print(
        f"test_month={window['test_month']} | "
        f"train={window['train_start']} -> {window['train_end']} | "
        f"train_shape={window['train_df'].shape} | "
        f"test_shape={window['test_df'].shape}"
    )

df_model shape: (139772, 36)
df_model period: 2022-01-01 01:00:00+01:00 -> 2025-12-31 23:45:00+01:00
n rolling windows: 12
test_month=2025-01 | train=2024-01 -> 2024-12 | train_shape=(35040, 36) | test_shape=(2976, 36)
test_month=2025-02 | train=2024-02 -> 2025-01 | train_shape=(35040, 36) | test_shape=(2688, 36)
test_month=2025-03 | train=2024-03 -> 2025-02 | train_shape=(34944, 36) | test_shape=(2972, 36)
test_month=2025-04 | train=2024-04 -> 2025-03 | train_shape=(34944, 36) | test_shape=(2880, 36)
test_month=2025-05 | train=2024-05 -> 2025-04 | train_shape=(34944, 36) | test_shape=(2976, 36)
test_month=2025-06 | train=2024-06 -> 2025-05 | train_shape=(34944, 36) | test_shape=(2880, 36)
test_month=2025-07 | train=2024-07 -> 2025-06 | train_shape=(34944, 36) | test_shape=(2976, 36)
test_month=2025-08 | train=2024-08 -> 2025-07 | train_shape=(34944, 36) | test_shape=(2976, 36)
test_month=2025-09 | train=2024-09 -> 2025-08 | train_shape=(34944, 36) | test_shape=(2880, 36)
test_month=20

In [19]:
# =========================================================
# 8. MODEL PIPELINES
# =========================================================

def package_model_result(model, scaler, X_val, y_val, y_val_pred_raw, y_test, y_pred_raw):
    bias_correction = compute_mean_bias(y_val, y_val_pred_raw)
    y_val_pred = apply_bias_correction(y_val_pred_raw, bias_correction)
    y_pred = apply_bias_correction(y_pred_raw, bias_correction)

    return {
        "model": model,
        "scaler": scaler,
        "bias_correction": bias_correction,
        "val_index": X_val.index.to_numpy(),
        "val_pred": y_val_pred,
        "val_pred_raw": y_val_pred_raw,
        "test_pred": y_pred,
        "test_pred_raw": y_pred_raw,
        "test_residuals": y_test.to_numpy() - y_pred,
        "test_mae": mean_absolute_error(y_test, y_pred),
        "test_rmse": rmse(y_test, y_pred),
    }


def run_lgb_pipeline(X_train, y_train, X_test, y_test, seed=SEED, valid_fraction=0.15):
    X_tr, y_tr, X_val, y_val = make_random_validation_split(
        X_train, y_train, valid_fraction=valid_fraction, seed=seed
    )

    model = lgb.LGBMRegressor(
        objective="mae",
        n_estimators=650,
        learning_rate=0.07,
        max_depth=10,
        num_leaves=95,
        min_child_samples=10,
        subsample=0.6,
        colsample_bytree=0.6,
        reg_alpha=0.01,
        reg_lambda=0.0,
        random_state=seed,
        n_jobs=-1,
    )

    model.fit(
        X_tr,
        y_tr,
        eval_set=[(X_val, y_val)],
        eval_metric="l1",
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)],
    )

    return package_model_result(
        model=model,
        scaler=None,
        X_val=X_val,
        y_val=y_val,
        y_val_pred_raw=model.predict(X_val),
        y_test=y_test,
        y_pred_raw=model.predict(X_test),
    )


def run_xgb_pipeline(X_train, y_train, X_test, y_test, seed=SEED, valid_fraction=0.15):
    X_tr, y_tr, X_val, y_val = make_random_validation_split(
        X_train, y_train, valid_fraction=valid_fraction, seed=seed
    )

    model = XGBRegressor(
        objective="reg:absoluteerror",
        eval_metric="mae",
        n_estimators=500,
        learning_rate=0.05,
        max_depth=8,
        min_child_weight=1,
        subsample=0.7,
        colsample_bytree=1.0,
        gamma=0.3,
        reg_alpha=0.01,
        random_state=seed,
        n_jobs=-1,
        tree_method="hist",
    )

    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)

    return package_model_result(
        model=model,
        scaler=None,
        X_val=X_val,
        y_val=y_val,
        y_val_pred_raw=model.predict(X_val),
        y_test=y_test,
        y_pred_raw=model.predict(X_test),
    )


def run_nn_pipeline(X_train, y_train, X_test, y_test, seed=SEED, valid_fraction=0.15, epochs=70, batch_size=128):
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)
    tf.keras.backend.clear_session()

    X_tr, y_tr, X_val, y_val = make_random_validation_split(
        X_train, y_train, valid_fraction=valid_fraction, seed=seed
    )

    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr)
    X_val_s = scaler.transform(X_val)
    X_test_s = scaler.transform(X_test)

    model = Sequential([
        Input(shape=(X_tr_s.shape[1],)),
        Dense(256, activation="relu", kernel_regularizer=l2(1e-6)),
        Dropout(0.2),
        Dense(128, activation="relu", kernel_regularizer=l2(1e-6)),
        Dropout(0.1),
        Dense(1),
    ])

    model.compile(optimizer=Adam(learning_rate=0.001), loss="mae")

    callbacks = [EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True)]

    model.fit(
        X_tr_s,
        y_tr,
        validation_data=(X_val_s, y_val),
        epochs=epochs,
        batch_size=batch_size,
        verbose=0,
        callbacks=callbacks,
        shuffle=False,
    )

    return package_model_result(
        model=model,
        scaler=scaler,
        X_val=X_val,
        y_val=y_val,
        y_val_pred_raw=model.predict(X_val_s, verbose=0).ravel(),
        y_test=y_test,
        y_pred_raw=model.predict(X_test_s, verbose=0).ravel(),
    )

In [20]:
# =========================================================
# 9. TRAIN SELECTED MODELS ACROSS ROLLING WINDOWS
# =========================================================

pipeline_map = {
    "lgb": run_lgb_pipeline,
    "xgb": run_xgb_pipeline,
    "nn": run_nn_pipeline,
}

results = {}

for model_name in selected_models:
    if model_name not in pipeline_map:
        raise ValueError(f"Unsupported model: {model_name}")

    print("" + "=" * 100)
    print(f"RUNNING {model_name.upper()} ACROSS ROLLING {test_year} MONTHS")
    print("=" * 100)

    monthly_runs = []
    test_pred_parts = []
    test_pred_raw_parts = []
    test_residual_parts = []
    test_actual_price_parts = []
    test_da_price_parts = []
    test_datetime_parts = []
    val_pred_parts = []
    val_pred_raw_parts = []
    val_actual_price_parts = []
    val_da_price_parts = []
    val_datetime_parts = []
    bias_parts = []

    for window in rolling_windows:
        train_window_df = window["train_df"]
        test_window_df = window["test_df"]

        X_train = train_window_df[feature_cols].copy()
        y_train = train_window_df[spread_col].copy()
        X_test = test_window_df[feature_cols].copy()
        y_test = test_window_df[spread_col].copy()

        with suppress_training_output():
            res = pipeline_map[model_name](X_train, y_train, X_test, y_test)

        val_df = train_window_df.loc[res["val_index"]].copy()

        monthly_runs.append({
            "test_month": window["test_month"],
            "train_start": window["train_start"],
            "train_end": window["train_end"],
            "train_rows": len(train_window_df),
            "test_rows": len(test_window_df),
            "val_rows": len(val_df),
            "bias_correction": res["bias_correction"],
        })

        test_pred_parts.append(np.asarray(res["test_pred"]))
        test_pred_raw_parts.append(np.asarray(res["test_pred_raw"]))
        test_residual_parts.append(np.asarray(res["test_residuals"]))
        test_actual_price_parts.append(test_window_df[target_col].to_numpy())
        test_da_price_parts.append(test_window_df["day_ahead_price"].to_numpy())
        test_datetime_parts.append(test_window_df["datetime"].to_numpy())

        val_pred_parts.append(np.asarray(res["val_pred"]))
        val_pred_raw_parts.append(np.asarray(res["val_pred_raw"]))
        val_actual_price_parts.append(val_df[target_col].to_numpy())
        val_da_price_parts.append(val_df["day_ahead_price"].to_numpy())
        val_datetime_parts.append(val_df["datetime"].to_numpy())
        bias_parts.append(res["bias_correction"])

        test_price_pred = test_window_df["day_ahead_price"].to_numpy() + np.asarray(res["test_pred"])
        test_price_metrics = evaluate_price_metrics(test_window_df[target_col].to_numpy(), test_price_pred)

        print(
            f"{window['test_month']} | "
            f"bias={res['bias_correction']:.6f} | "
            f"test_MAE={test_price_metrics['MAE']:.6f} | "
            f"test_RMSE={test_price_metrics['RMSE']:.6f}"
        )

    results[model_name] = {
        "feature_cols": feature_cols,
        "monthly_runs": monthly_runs,
        "test_pred": np.concatenate(test_pred_parts),
        "test_pred_raw": np.concatenate(test_pred_raw_parts),
        "test_residuals": np.concatenate(test_residual_parts),
        "test_actual_price": np.concatenate(test_actual_price_parts),
        "test_da_price": np.concatenate(test_da_price_parts),
        "test_datetime": np.concatenate(test_datetime_parts),
        "val_pred": np.concatenate(val_pred_parts),
        "val_pred_raw": np.concatenate(val_pred_raw_parts),
        "val_actual_price": np.concatenate(val_actual_price_parts),
        "val_da_price": np.concatenate(val_da_price_parts),
        "val_datetime": np.concatenate(val_datetime_parts),
        "bias_corrections": bias_parts,
    }

RUNNING LGB ACROSS ROLLING 2025 MONTHS
2025-01 | bias=-0.762004 | test_MAE=32.719369 | test_RMSE=52.071683
2025-02 | bias=-2.029794 | test_MAE=28.871648 | test_RMSE=49.924140
2025-03 | bias=0.399756 | test_MAE=31.777724 | test_RMSE=47.889550
2025-04 | bias=0.855196 | test_MAE=33.654413 | test_RMSE=62.523237
2025-05 | bias=0.796638 | test_MAE=30.477629 | test_RMSE=50.079861
2025-06 | bias=-0.732878 | test_MAE=50.514653 | test_RMSE=147.074102
2025-07 | bias=0.438021 | test_MAE=28.697134 | test_RMSE=60.617296
2025-08 | bias=-1.774979 | test_MAE=32.718928 | test_RMSE=51.858524
2025-09 | bias=-0.116818 | test_MAE=30.935945 | test_RMSE=69.588415
2025-10 | bias=1.239653 | test_MAE=24.454055 | test_RMSE=45.008384
2025-11 | bias=-0.043913 | test_MAE=35.522153 | test_RMSE=66.317445
2025-12 | bias=-1.947891 | test_MAE=18.439481 | test_RMSE=28.576720
RUNNING XGB ACROSS ROLLING 2025 MONTHS
2025-01 | bias=-1.366571 | test_MAE=28.928981 | test_RMSE=42.672690
2025-02 | bias=-2.040508 | test_MAE=29.080

In [25]:
# =========================================================
# 10. COMPARE VALIDATION AND TEST METRICS
# =========================================================

model_order = ["lgb", "xgb", "nn"]

validation_rows = []
test_rows = []

first_res = next(iter(results.values()))

validation_rows.append({
    "Model": "naive",
    **evaluate_price_metrics(
        first_res["val_actual_price"],
        first_res["val_da_price"],
    ),
})

test_rows.append({
    "Model": "naive",
    **evaluate_price_metrics(
        first_res["test_actual_price"],
        first_res["test_da_price"],
    ),
})


for model_name in model_order:

    if model_name not in results:
        continue

    res = results[model_name]

    validation_rows.append({
        "Model": model_name,
        **evaluate_price_metrics(
            res["val_actual_price"],
            res["val_da_price"] + res["val_pred"],
        ),
    })

    test_rows.append({
        "Model": model_name,
        **evaluate_price_metrics(
            res["test_actual_price"],
            res["test_da_price"] + res["test_pred"],
        ),
    })


validation_df = format_metric_table(
    pd.DataFrame(validation_rows)
)

test_df_metrics = format_metric_table(
    pd.DataFrame(test_rows)
)


print("\n" + "=" * 100)
print("VALIDATION METRICS")
print("=" * 100)
print(validation_df.to_string(index=False))


print("\n\n")

print("=" * 100)
print(f"TEST METRICS (ROLLING {test_year} MONTHS)")
print("=" * 100)
print(test_df_metrics.to_string(index=False))


VALIDATION METRICS
Model  ResidualMean       MAE      RMSE       R2  p95_abs_residual  p99_abs_residual
naive -2.951092e-01 34.929765 99.918398 0.214916         96.222000        230.561200
  lgb -1.119943e-16 23.708257 86.144520 0.416447         65.718853        155.951856
  xgb -5.184547e-08 24.055813 84.086447 0.443997         66.687044        157.129955
   nn -1.107402e-08 27.268415 93.529570 0.312104         71.768271        188.408784



TEST METRICS (ROLLING 2025 MONTHS)
Model  ResidualMean       MAE      RMSE       R2  p95_abs_residual  p99_abs_residual
naive      0.082428 31.226034 66.303141 0.356545         91.608500        195.340900
  lgb      6.201504 31.555743 66.931656 0.344288         91.367318        229.103153
  xgb      4.017669 30.661204 64.987182 0.381833         89.206223        195.876891
   nn      0.943196 29.117885 63.122544 0.416798         77.855879        181.172855
